# GPU Memory Subsystem: Registers, Shared Memory/L1, L2, and HBM

This notebook introduces the GPU memory hierarchy and why it matters for performance.

In modern deep learning systems—especially during LLM decoding—memory bandwidth is often the primary bottleneck, not compute.

## 1. The GPU Memory Hierarchy at a Glance

The memory hierarchy follows a simple hardware rule: the faster the memory, the smaller its capacity and the closer it must sit to the ALUs.

```text
REGISTERS (Fastest, Smallest)
~256 KB per SM | Latency: ~0–1 cycles | Bandwidth: ~30–40 TB/s aggregate

└───────────────────────────────┬──────────────────────────────────────┘
                                │
SHARED MEMORY / L1 DATA CACHE (On-Chip SRAM)
~128 KB–228 KB per SM | Latency: ~20–30 cycles | Bandwidth: ~15–20 TB/s

└───────────────────────────────┬──────────────────────────────────────┘
                                │
L2 CACHE (Die-Level Shared SRAM)
~40 MB–50 MB per GPU | Latency: ~150–200 cycles | Bandwidth: ~5–7 TB/s

└───────────────────────────────┬──────────────────────────────────────┘
                                │
HIGH-BANDWIDTH MEMORY / HBM (Off-Chip DRAM / VRAM)
80 GB–144 GB | Latency: ~400–800 cycles | Bandwidth: 2.0–3.35 TB/s

└───────────────────────────────┬──────────────────────────────────────┘
                                │
HOST SYSTEM MEMORY (CPU RAM via PCIe Gen5 / NVLink C2C)
512 GB–2 TB | Latency: thousands of cycles | Bandwidth: ~64–128 GB/s
```


## 2. Quantitative Reality: Latency and Bandwidth Comparison

| Memory level | Location | Scope | Typical size | Latency | Bandwidth |
| --- | --- | --- | --- | --- | --- |
| Registers | On-chip inside SM | Private to one thread | ~256 KB per SM | ~1 cycle | ~30+ TB/s |
| Shared Memory / L1 | On-chip inside SM | Shared across one block | Up to 228 KB per SM | ~20–30 cycles | ~15–20 TB/s |
| L2 Cache | On-die shared SRAM | Accessible by all SMs | ~50 MB total | ~150–200 cycles | ~6 TB/s |
| Global Memory (HBM3) | Off-die on substrate | Accessible by the whole grid and host | 80 GB | ~400–800 cycles | ~3.35 TB/s |
| Host RAM | CPU motherboard | CPU DRAM | 512 GB+ | >2000 cycles | ~64 GB/s over PCIe 5.0 |

The gap is enormous:

- moving from registers to HBM adds a roughly 400× to 800× latency penalty,
- moving from HBM to CPU RAM over PCIe causes another large bandwidth drop.

## Memory

## 3. Tier 1: The Register File

The register file is the fastest memory in the GPU. It sits directly in the datapath of the ALUs inside each SM sub-core.

### Physical structure

Registers are implemented as SRAM and are allocated per thread when the kernel is compiled.

### Why it matters

Each thread gets its own private register space. This is where temporary values and frequently used variables live.

### The engineering bottleneck: register spilling

Each SM has a hard limit on the number of 32-bit registers it can hold. If a kernel uses too many variables, the compiler spills some values to local memory instead.

> This is a major performance problem because local memory is not truly fast on-chip storage; it is backed by slow global memory.



## 4. Tier 2: Shared Memory and L1 Cache

In modern GPUs, shared memory and L1 data cache share the same physical on-chip SRAM pool inside each SM.

```text
SM COMBINED SRAM POOL
┌──────────────────────────────────────┬────────────────────┐
│ Shared Memory (Software Controlled)  │ L1 Data Cache      │
│ Accessible by all threads in block   │ Hardware managed   │
└──────────────────────────────────────┴────────────────────┘
```

### Shared memory

Shared memory is a software-managed scratchpad. It is explicitly used by the programmer to reuse data within a block.

### Why it exists

If many threads need the same data from global memory, loading it repeatedly wastes bandwidth. Shared memory allows the block to load the tile once and reuse it many times.

## 5. Tier 3: L2 Cache

L2 cache is a large shared SRAM pool on the GPU die. It sits between the SMs and the off-chip HBM and serves as the main coherence and caching layer for memory traffic.

It helps with:

- caching reads and writes from SMs,
- coordinating memory accesses across thread blocks,
- keeping frequently used data available close to the compute cores.

## 6. Tier 4: High-Bandwidth Memory (HBM3 / Global VRAM)

When people say a model requires an 80 GB GPU, they are referring to the amount of global memory available in HBM.

```text
GPU DIE
┌───────────────┐ ┌───────────────┐ ┌───────────────┐ ┌──────────────┐
│     GPC 0     │ │     GPC 1     │ │     GPC 2     │ │    GPC 3     │
└───────────────┘ └───────────────┘ └───────────────┘ └──────────────┘
          │               │               │               │
          └───────────────┬───────────────────────────────┘
                          │
                 SILICON INTERPOSER
                          │
        ┌─────────────────────┼─────────────────────┐
        │                     │                     │
   HBM3 Stack 0         HBM3 Stack 1         HBM3 Stack 2
```

HBM is used because it provides far higher memory bandwidth than standard CPU DDR memory. That makes it suitable for large AI models and memory-intensive workloads.

## 7. Memory Coalescing

Global memory transfers happen in cache-line-sized chunks such as 32, 64, or 128 bytes.

### Scenario A: Coalesced access

When the 32 threads in a warp access contiguous memory addresses, the hardware can combine them into a single efficient transaction.

```text
Threads in warp: [T0] [T1] [T2] [T3] ... [T31]
Addresses:       [0x00] [0x04] [0x08] [0x0C] ... [0x7C]

Result: one efficient 128-byte memory transaction
```

### Scenario B: Strided or uncoalesced access

If the threads access scattered addresses, the hardware must issue many separate transactions, greatly reducing memory efficiency.

```text
Threads in warp: [T0] [T1] [T2] ... [T31]
Addresses:       [0x00] [0x400] [0x800] ... [0x7C00]

Result: 32 separate transactions instead of one
```

Coalescing is critical for LLM inference and other bandwidth-bound workloads because poor access patterns can waste most of the memory bus bandwidth.